In [ ]:
import time
import json
from pathlib import Path
import pandas as pd
from pyalex import Authors, config

config.email = "arav_kumar@berkeley.edu"  

In [6]:
def fetch_seed_publications(openalex_id, from_year=2015, to_year=2025): 
    """Return all publications by one seed author, flattened to one row per (paper, author)."""
    rows = []
    works = Works().filter(
        author={"id": f"https://openalex.org/{openalex_id}"}, 
        publication_year=f"{from_year}-{to_year}"
    )
    
    for page in works.paginate(per_page=200, n_max=None):
        for w in page:
            primary_topic = w.get("primary_topic") or {}
            
            for authorship in w["authorships"]:
                author = authorship["author"]
                if not author.get("id"):
                    continue
                
                institutions = authorship["institutions"]
                if not institutions:
                    institutions = [{}]
                
                for inst in institutions:
                    rows.append({
                        "pub_id": w["id"].replace("https://openalex.org/", ""),
                        "doi": (w.get("doi") or "").replace("https://doi.org/", ""),
                        "pub_date": w.get("publication_date"),
                        "author_openalex_id": author["id"].replace("https://openalex.org/", ""),
                        "display_name": author["display_name"],
                        "aff_country_code": inst.get("country_code"),
                        "aff_name": inst.get("display_name"),
                        "subfield": (primary_topic.get("subfield") or {}).get("display_name"),
                        "field": (primary_topic.get("field") or {}).get("display_name"),
                        "domain": (primary_topic.get("domain") or {}).get("display_name"),
                        "citations": w.get("cited_by_count"),
                    })
    
    return pd.DataFrame(rows)

In [7]:
from pathlib import Path

seeds = pd.read_csv("seed_authors_for_matching.csv")

all_frames = []
for i, row in seeds.iterrows():
    oid = row["openalex_id"]
    seed_label = row["author_anon_id"]
    print(f"[{i+1}/{len(seeds)}] Fetching {seed_label} ({row['first_name']} {row['last_name']}, {oid})...")
    df = fetch_seed_publications(oid, 2015, 2025)
    df["seed_author_anon_id"] = seed_label
    all_frames.append(df)
    df.to_csv(f"extended_pull_{seed_label}.csv", index=False)  # incremental save
    print(f"  → {len(df)} rows")

combined = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal rows across all seeds: {len(combined)}")

# Deduplicate on (pub_id, author_openalex_id, aff_country_code) 
# because two seeds who co-authored a paper produce duplicate rows
combined = combined.drop_duplicates(
    subset=["pub_id", "author_openalex_id", "aff_country_code"],
    keep="first"
).reset_index(drop=True)
print(f"After deduplication: {len(combined)} rows")

combined.to_csv("extended_cohort_raw.csv", index=False)
print(f"Wrote extended_cohort_raw.csv")

[1/10] Fetching A38327606686f (Hermenegildo García, A5014540719)...
  → 6059 rows
[2/10] Fetching Ab1e8b19bae0b (Jiaguo Yu, A5100423139)...
  → 4849 rows
[3/10] Fetching A11460d5aa670 (Bicheng Zhu, A5032509793)...
  → 1205 rows
[4/10] Fetching Af103ef0d80ce (Liuyang Zhang, A5101925067)...
  → 1271 rows
[5/10] Fetching Afb5914645166 (Jianjun Zhang, A5100358209)...
  → 558 rows
[6/10] Fetching A60af1130add6 (Chuanjia Jiang, A5056638329)...
  → 772 rows
[7/10] Fetching A6e2aff8d4f11 (Junwei Fu, A5013848651)...
  → 2553 rows
[8/10] Fetching Aa6270b01971b (Liyuan Chai, A5079893090)...
  → 4286 rows
[9/10] Fetching A1d51b8d97097 (Johannes F E Mann, A5043842967)...
  → 4224 rows
[10/10] Fetching A77c21dacbd16 (Rajiv Agarwal, A5089305017)...
  → 4958 rows

Total rows across all seeds: 30735
After deduplication: 22261 rows
Wrote extended_cohort_raw.csv


In [8]:
print("Unique papers:", combined["pub_id"].nunique())
print("Unique authors:", combined["author_openalex_id"].nunique())
print("\nRows per seed (should roughly reflect each seed's productivity):")
print(combined.groupby("seed_author_anon_id").size())
print("\nCountry distribution (top 10):")
print(combined["aff_country_code"].value_counts().head(10))
print("\nSubfield distribution (top 10):")
print(combined["subfield"].value_counts().head(10))

Unique papers: 2549
Unique authors: 6722

Rows per seed (should roughly reflect each seed's productivity):
seed_author_anon_id
A11460d5aa670     459
A1d51b8d97097    3181
A38327606686f    4078
A60af1130add6     435
A6e2aff8d4f11    2105
A77c21dacbd16    3895
Aa6270b01971b    2702
Ab1e8b19bae0b    4102
Af103ef0d80ce    1117
Afb5914645166     187
dtype: int64

Country distribution (top 10):
aff_country_code
CN    8724
US    2168
ES    1874
DE     853
GB     450
AU     415
SA     337
DK     329
CA     298
FR     245
Name: count, dtype: int64

Subfield distribution (top 10):
subfield
Renewable Energy, Sustainability and the Environment    5564
Endocrinology, Diabetes and Metabolism                  2921
Materials Chemistry                                     2284
Inorganic Chemistry                                     1125
Cardiology and Cardiovascular Medicine                   939
Nephrology                                               912
Electrical and Electronic Engineering          

In [21]:
from commindex import make_topic_key 
def adapt_extended_for_pipeline(df, author_topics_path="author_topics.json"):
    import json
    with open(author_topics_path) as f:
        author_topics = json.load(f)
    
    df = df.copy()
    
    def get_specialization(row):
        # Try author-level topic first
        aid = row["author_openalex_id"]
        topic = author_topics.get(aid)
        if topic is not None:
            return make_topic_key(topic["subfield"], topic["field"], topic["domain"])
        # Fall back to paper-level topic
        if pd.notna(row["subfield"]):
            return make_topic_key(row["subfield"], row["field"], row["domain"])
        return None
    
    df["specialization"] = df.apply(get_specialization, axis=1)
    
    df = df.rename(columns={
        "pub_id": "pub_anon_id",
        "author_openalex_id": "author_anon_id",
    })
    df["gender"] = "unknown"
    return df

In [23]:
combined = pd.read_csv("extended_cohort_raw.csv")

In [24]:
adapted_extended_cohort = adapt_extended_for_pipeline(combined)
adapted_extended_cohort.to_csv("adapted_extended_cohort.csv", index=False)

In [ ]:
def test_country_coverage(df, lookup_col="aff_country_code"):
    """Check every country code in df is covered by INCOME_GROUP_LOOKUP."""
    from commindex import INCOME_GROUP_LOOKUP
    
    countries_in_data = set(df[lookup_col].dropna().unique())
    countries_in_lookup = set(INCOME_GROUP_LOOKUP.keys())
    missing = countries_in_data - countries_in_lookup
    
    if missing:
        # Show how many rows are affected by each missing country
        counts = df[df[lookup_col].isin(missing)][lookup_col].value_counts()
        raise AssertionError(
            f"{len(missing)} country codes not in INCOME_GROUP_LOOKUP:\n{counts}"
        )
    
    print(f"✓ All {len(countries_in_data)} country codes covered by lookup")

In [9]:
import pandas as pd
df = pd.read_csv("adapted_extended_cohort.csv")
test_country_coverage(df)

✓ All 81 country codes covered by lookup


In [10]:
df.columns

Index(['pub_anon_id', 'doi', 'pub_date', 'author_anon_id', 'display_name',
       'aff_country_code', 'aff_name', 'subfield', 'field', 'domain',
       'citations', 'seed_author_anon_id', 'specialization', 'gender'],
      dtype='str')

In [11]:
n = len(df)
missing_country = df["aff_country_code"].isna().sum()
missing_discipline = df["specialization"].isna().sum()
missing_both = df[df["aff_country_code"].isna() & df["specialization"].isna()].shape[0]

print(f"Total rows: {n:,}")
print(f"Missing country only: {missing_country - missing_both:,} ({100*(missing_country-missing_both)/n:.1f}%)")
print(f"Missing discipline only: {missing_discipline - missing_both:,} ({100*(missing_discipline-missing_both)/n:.1f}%)")
print(f"Missing both: {missing_both:,} ({100*missing_both/n:.1f}%)")

Total rows: 22,261
Missing country only: 4,409 (19.8%)
Missing discipline only: 0 (0.0%)
Missing both: 0 (0.0%)


In [12]:
# Identify the missing-country rows
missing = df[df["aff_country_code"].isna()]
print(f"Missing-country rows: {len(missing):,}")

# For each (paper, author) with a missing-country row, check if OTHER rows exist with a valid country
missing_pairs = missing[["pub_anon_id", "author_anon_id"]].drop_duplicates()
print(f"Unique (paper, author) pairs with a missing-country row: {len(missing_pairs):,}")

# Now check how many of those pairs ALSO have a valid country row elsewhere
def has_valid_country_elsewhere(row):
    other_rows = df[
        (df["pub_anon_id"] == row["pub_anon_id"]) &
        (df["author_anon_id"] == row["author_anon_id"]) &
        (df["aff_country_code"].notna())
    ]
    return len(other_rows) > 0

missing_pairs["has_alt_country"] = missing_pairs.apply(has_valid_country_elsewhere, axis=1)
n_recoverable = missing_pairs["has_alt_country"].sum()
n_lost = len(missing_pairs) - n_recoverable

print(f"\nOf those {len(missing_pairs):,} (paper, author) pairs:")
print(f"  {n_recoverable:,} have at least one other row with a valid country (recoverable)")
print(f"  {n_lost:,} would lose the author from that paper entirely (unrecoverable)")

# Impact on seeds specifically
seeds = pd.read_csv("seed_authors_for_matching.csv")
seed_ids = set(seeds["author_anon_id"])
lost_pairs = missing_pairs[~missing_pairs["has_alt_country"]]
seed_impact = lost_pairs[lost_pairs["author_anon_id"].isin(seed_ids)]
print(f"\nSeed authors affected by unrecoverable losses: {len(seed_impact):,} (paper, author) pairs")

Missing-country rows: 4,409
Unique (paper, author) pairs with a missing-country row: 4,409

Of those 4,409 (paper, author) pairs:
  821 have at least one other row with a valid country (recoverable)
  3,588 would lose the author from that paper entirely (unrecoverable)

Seed authors affected by unrecoverable losses: 0 (paper, author) pairs


In [13]:
missing = df[df["aff_country_code"].isna()]
print("Sample of missing-country rows:")
print(missing[["pub_anon_id", "author_anon_id", "display_name", "aff_name"]].head(20).to_string())
print()
print("How many missing-country rows have an aff_name populated?")
print(f"  {missing['aff_name'].notna().sum():,} of {len(missing):,}")

Sample of missing-country rows:
      pub_anon_id author_anon_id                           display_name aff_name
276   W2762636073    A5091536895                          Nuria Bermejo      NaN
725   W4287225839    A5078077932                            S. Barzilai      NaN
964   W4404849857    A5014355779                  Rodrigo García‐Muelas      NaN
986   W4410090578    A5000230513  Mohammad Mehdi Moftakhari Sharifzadeh      NaN
1205  W2237123976    A5036370021                            Andrés Vega      NaN
2264  W1898428355    A5074291981                         G. Deligeorgis      NaN
2277  W4399654853    A5053454388               Andrés Uscategui-Linares      NaN
2278  W4399654853    A5011494070               Andrea Santiago‐Portillo      NaN
2284  W4399654853    A5014540719                   Hermenegildo Garcı́a      NaN
2347  W4396767409    A5029865747               Francisco Bernat-Quesada      NaN
2356  W4404918307    A5008067318                  Daniel Vázquez‐Tarrío      

In [15]:
df = pd.read_csv("adapted_extended_cohort.csv", parse_dates=["pub_date"])
missing = df[df["aff_country_code"].isna()]
paper_missing_counts = missing.groupby("pub_anon_id").size()
paper_total_counts = df.groupby("pub_anon_id").size()

# What fraction of each affected paper is missing country?
fraction_missing = paper_missing_counts / paper_total_counts.loc[paper_missing_counts.index]
print(f"Papers with ANY missing country: {len(paper_missing_counts):,}")
print(f"Papers where ALL rows are missing country: {(fraction_missing == 1.0).sum():,}")
print(f"Papers where >50% of rows are missing country: {(fraction_missing > 0.5).sum():,}")

# Are these mostly older papers?
paper_dates = df.groupby("pub_anon_id")["pub_date"].first()
missing_paper_dates = paper_dates.loc[paper_missing_counts.index]
print(f"\nMissing-country papers by year:")
print(missing_paper_dates.dt.year.value_counts().sort_index())

Papers with ANY missing country: 661
Papers where ALL rows are missing country: 230
Papers where >50% of rows are missing country: 276

Missing-country papers by year:
pub_date
2015    59
2016    70
2017    66
2018    47
2019    47
2020    69
2021    53
2022    76
2023    59
2024    45
2025    70
Name: count, dtype: int64


In [17]:
before = pd.read_csv("adapted_extended_cohort.csv", parse_dates=["pub_date"])
after = before.dropna(subset=["aff_country_code", "specialization"])

seeds = pd.read_csv("seed_authors_for_matching.csv")
for _, row in seeds.iterrows():
    oid = row["openalex_id"]
    n_before = before[before["author_anon_id"] == oid]["pub_anon_id"].nunique()
    n_after = after[after["author_anon_id"] == oid]["pub_anon_id"].nunique()
    pct = 100 * n_after / n_before if n_before else 0
    marker = "" if pct >= 80 else "  ← significant loss"
    print(f"{oid} ({row['first_name']} {row['last_name']}): {n_after}/{n_before} papers ({pct:.0f}%){marker}")

A5014540719 (Hermenegildo García): 492/565 papers (87%)
A5100423139 (Jiaguo Yu): 615/660 papers (93%)
A5032509793 (Bicheng Zhu): 159/162 papers (98%)
A5101925067 (Liuyang Zhang): 175/197 papers (89%)
A5100358209 (Jianjun Zhang): 72/76 papers (95%)
A5056638329 (Chuanjia Jiang): 105/110 papers (95%)
A5013848651 (Junwei Fu): 154/156 papers (99%)
A5079893090 (Liyuan Chai): 341/363 papers (94%)
A5043842967 (Johannes F E Mann): 126/179 papers (70%)  ← significant loss
A5089305017 (Rajiv Agarwal): 242/300 papers (81%)


In [18]:
def fetch_author_topics(openalex_id):
    """Return the primary subfield/field/domain for one author."""
    try:
        a = Authors()[f"https://openalex.org/{openalex_id}"]
    except Exception as e:
        return None
    
    topics = a.get("topics") or []
    if not topics:
        return None
    
    top = topics[0]  # highest-ranked topic
    return {
        "subfield": (top.get("subfield") or {}).get("display_name"),
        "field": (top.get("field") or {}).get("display_name"),
        "domain": (top.get("domain") or {}).get("display_name"),
    }

In [20]:
extended = pd.read_csv("extended_cohort_raw.csv")
unique_authors = extended["author_openalex_id"].unique()
print(f"Fetching topics for {len(unique_authors):,} authors...")

author_topics = {}
for i, aid in enumerate(unique_authors):
    if i % 100 == 0:
        print(f"  [{i}/{len(unique_authors)}]")
    author_topics[aid] = fetch_author_topics(aid)

# Save for reuse (so you don't have to re-query)
import json
with open("author_topics.json", "w") as f:
    json.dump({k: v for k, v in author_topics.items() if v}, f, indent=2)

Fetching topics for 6,722 authors...
  [0/6722]
  [100/6722]
  [200/6722]
  [300/6722]
  [400/6722]
  [500/6722]
  [600/6722]
  [700/6722]
  [800/6722]
  [900/6722]
  [1000/6722]
  [1100/6722]
  [1200/6722]
  [1300/6722]
  [1400/6722]
  [1500/6722]
  [1600/6722]
  [1700/6722]
  [1800/6722]
  [1900/6722]
  [2000/6722]
  [2100/6722]
  [2200/6722]
  [2300/6722]
  [2400/6722]
  [2500/6722]
  [2600/6722]
  [2700/6722]
  [2800/6722]
  [2900/6722]
  [3000/6722]
  [3100/6722]
  [3200/6722]
  [3300/6722]
  [3400/6722]
  [3500/6722]
  [3600/6722]
  [3700/6722]
  [3800/6722]
  [3900/6722]
  [4000/6722]
  [4100/6722]
  [4200/6722]
  [4300/6722]
  [4400/6722]
  [4500/6722]
  [4600/6722]
  [4700/6722]
  [4800/6722]
  [4900/6722]
  [5000/6722]
  [5100/6722]
  [5200/6722]
  [5300/6722]
  [5400/6722]
  [5500/6722]
  [5600/6722]
  [5700/6722]
  [5800/6722]
  [5900/6722]
  [6000/6722]
  [6100/6722]
  [6200/6722]
  [6300/6722]
  [6400/6722]
  [6500/6722]
  [6600/6722]
  [6700/6722]


In [25]:
adapted = pd.read_csv("adapted_extended_cohort.csv")
n_missing = adapted["specialization"].isna().sum()
print(f"Rows with missing specialization: {n_missing:,} of {len(adapted):,} ({100*n_missing/len(adapted):.1f}%)")

# How many are because author had no topic AND paper had no subfield?
raw = pd.read_csv("extended_cohort_raw.csv")
# Match rows: pub_id → pub_anon_id, author_openalex_id → author_anon_id
raw_no_topic = raw["subfield"].isna().sum()
print(f"Raw rows with no paper subfield: {raw_no_topic:,}")

Rows with missing specialization: 230 of 22,261 (1.0%)
Raw rows with no paper subfield: 335


In [26]:
# For a paper with a specific specialization value, how many authors have that same value?
sample_paper = adapted.groupby("pub_anon_id").size().sort_values(ascending=False).index[0]
paper_rows = adapted[adapted["pub_anon_id"] == sample_paper]
print(f"Paper {sample_paper}: {len(paper_rows)} rows, unique specializations: {paper_rows['specialization'].nunique()}")
print(paper_rows["specialization"].value_counts())

Paper W3200915536: 108 rows, unique specializations: 1
specialization
Health Sciences||Medicine||Endocrinology, Diabetes and Metabolism    108
Name: count, dtype: int64


In [27]:
import json
import pandas as pd

with open("author_topics.json") as f:
    author_topics = json.load(f)

adapted = pd.read_csv("adapted_extended_cohort.csv")
unique_authors = set(adapted["author_anon_id"].unique())
authors_with_topic = set(author_topics.keys())
authors_without = unique_authors - authors_with_topic

print(f"Total unique authors: {len(unique_authors):,}")
print(f"Authors with OpenAlex.topics profile: {len(authors_with_topic):,}")
print(f"Authors WITHOUT: {len(authors_without):,} ({100*len(authors_without)/len(unique_authors):.1f}%)")

Total unique authors: 6,722
Authors with OpenAlex.topics profile: 972
Authors WITHOUT: 5,750 (85.5%)


In [28]:
# Paper counts by author
paper_counts = adapted.groupby("author_anon_id")["pub_anon_id"].nunique()

# Categorize
multi_paper_authors = set(paper_counts[paper_counts >= 2].index)
single_paper_authors = set(paper_counts[paper_counts == 1].index)

missing_multi = authors_without & multi_paper_authors
missing_single = authors_without & single_paper_authors

print(f"\nMulti-paper authors: {len(multi_paper_authors):,}")
print(f"  Missing topic profile: {len(missing_multi):,} ({100*len(missing_multi)/len(multi_paper_authors):.1f}%)")
print(f"\nSingle-paper authors: {len(single_paper_authors):,}")
print(f"  Missing topic profile: {len(missing_single):,} ({100*len(missing_single)/len(single_paper_authors):.1f}%)")


Multi-paper authors: 2,469
  Missing topic profile: 2,057 (83.3%)

Single-paper authors: 4,253
  Missing topic profile: 3,693 (86.8%)


In [29]:
# Per paper, count characterized authors (those with own topic)
paper_char_counts = adapted[adapted["author_anon_id"].isin(authors_with_topic)].groupby("pub_anon_id")["author_anon_id"].nunique()

# How many papers have <2 characterized authors?
total_papers = adapted["pub_anon_id"].nunique()
papers_with_lt2_char = (paper_char_counts < 2).sum() + (total_papers - len(paper_char_counts))
print(f"\nTotal papers: {total_papers:,}")
print(f"Papers with <2 characterized authors (discipline undefined): {papers_with_lt2_char:,} ({100*papers_with_lt2_char/total_papers:.1f}%)")


Total papers: 2,549
Papers with <2 characterized authors (discipline undefined): 1,553 (60.9%)


In [30]:
seeds = pd.read_csv("seed_authors_for_matching.csv")
print("\nPer-seed papers with sufficient discipline data (≥2 characterized authors):")
for _, row in seeds.iterrows():
    oid = row["openalex_id"]
    seed_papers = adapted[adapted["author_anon_id"] == oid]["pub_anon_id"].unique()
    usable = sum(paper_char_counts.get(pid, 0) >= 2 for pid in seed_papers)
    pct = 100 * usable / len(seed_papers) if len(seed_papers) else 0
    marker = "" if pct >= 70 else "  ← low coverage"
    print(f"  {row['first_name']} {row['last_name']}: {usable}/{len(seed_papers)} papers ({pct:.0f}%){marker}")


Per-seed papers with sufficient discipline data (≥2 characterized authors):
  Hermenegildo García: 560/565 papers (99%)
  Jiaguo Yu: 386/660 papers (58%)  ← low coverage
  Bicheng Zhu: 127/162 papers (78%)
  Liuyang Zhang: 5/197 papers (3%)  ← low coverage
  Jianjun Zhang: 56/76 papers (74%)
  Chuanjia Jiang: 53/110 papers (48%)  ← low coverage
  Junwei Fu: 8/156 papers (5%)  ← low coverage
  Liyuan Chai: 0/363 papers (0%)  ← low coverage
  Johannes F E Mann: 0/179 papers (0%)  ← low coverage
  Rajiv Agarwal: 0/300 papers (0%)  ← low coverage


In [31]:
print(f"Entries in author_topics.json: {len(author_topics)}")
print("Sample keys:")
print(list(author_topics.keys())[:5])

Entries in author_topics.json: 972
Sample keys:
['A5047867173', 'A5100376706', 'A5014540719', 'A5087568651', 'A5020061120']


In [9]:
from pyalex import Authors
a = Authors()["https://openalex.org/A5089305017"]  # Rajiv Agarwal
print("Topics:", a.get("topics", "none")[:3] if a.get("topics") else "empty")

Topics: [{'id': 'https://openalex.org/T10144', 'display_name': 'Blood Pressure and Hypertension Studies', 'count': 114, 'subfield': {'id': 'https://openalex.org/subfields/2705', 'display_name': 'Cardiology and Cardiovascular Medicine'}, 'field': {'id': 'https://openalex.org/fields/27', 'display_name': 'Medicine'}, 'domain': {'id': 'https://openalex.org/domains/4', 'display_name': 'Health Sciences'}}, {'id': 'https://openalex.org/T10291', 'display_name': 'Dialysis and Renal Disease Management', 'count': 79, 'subfield': {'id': 'https://openalex.org/subfields/2727', 'display_name': 'Nephrology'}, 'field': {'id': 'https://openalex.org/fields/27', 'display_name': 'Medicine'}, 'domain': {'id': 'https://openalex.org/domains/4', 'display_name': 'Health Sciences'}}, {'id': 'https://openalex.org/T10408', 'display_name': 'Chronic Kidney Disease and Diabetes', 'count': 55, 'subfield': {'id': 'https://openalex.org/subfields/2727', 'display_name': 'Nephrology'}, 'field': {'id': 'https://openalex.org

In [10]:
extended = pd.read_csv("extended_cohort_raw.csv")

unique_authors = extended["author_openalex_id"].unique()

In [11]:
import time
import json
from pathlib import Path
import pandas as pd
from pyalex import Authors, config

# CRITICAL: set polite pool email BEFORE any queries
config.email = "arav_kumar@berkeley.edu"


def fetch_author_batch(openalex_ids):
    """Fetch up to 50 authors in one query via pipe-separated filter."""
    id_filter = "|".join(openalex_ids)
    authors = Authors().filter(openalex_id=id_filter).get(per_page=len(openalex_ids))
    return authors


def extract_author_metadata(author_obj):
    """Extract topic + h-index from one Author response object."""
    topics = author_obj.get("topics") or []
    top = topics[0] if topics else None
    return {
        "subfield": (top.get("subfield") or {}).get("display_name") if top else None,
        "field": (top.get("field") or {}).get("display_name") if top else None,
        "domain": (top.get("domain") or {}).get("display_name") if top else None,
        "h_index": (author_obj.get("summary_stats") or {}).get("h_index"),
        "works_count": author_obj.get("works_count"),
        "cited_by_count": author_obj.get("cited_by_count"),
    }


def fetch_all_authors_batched(ids_to_fetch, existing=None, batch_size=50, sleep_between=0.2):
    """Batched fetch with resume support. Saves progress every 20 batches."""
    results = dict(existing) if existing else {}
    output_path = Path("author_topics.json")
    
    total = len(ids_to_fetch)
    for i in range(0, total, batch_size):
        batch = ids_to_fetch[i:i+batch_size]
        try:
            authors = fetch_author_batch(batch)
            fetched_ids = set()
            for a in authors:
                aid = a["id"].replace("https://openalex.org/", "")
                results[aid] = extract_author_metadata(a)
                fetched_ids.add(aid)
            
            # Mark any batch member NOT returned as null (missing from OpenAlex)
            for aid in batch:
                if aid not in fetched_ids:
                    results[aid] = None
        except Exception as e:
            print(f"  batch {i//batch_size} failed: {e}")
            # Don't crash — keep going
        
        # Progress + incremental save
        if (i // batch_size) % 20 == 0:
            with open(output_path, "w") as f:
                json.dump({k: v for k, v in results.items() if v is not None}, f, indent=2)
            print(f"  [{i:5}/{total}] saved checkpoint ({len(results):,} total)")
        
        time.sleep(sleep_between)
    
    # Final save
    with open(output_path, "w") as f:
        json.dump({k: v for k, v in results.items() if v is not None}, f, indent=2)
    print(f"Done. Total authors with metadata: {sum(1 for v in results.values() if v is not None):,}")
    return results


# --- Main run ---
# Load existing (resume support)
try:
    with open("author_topics.json") as f:
        existing_topics = json.load(f)
    print(f"Resuming from {len(existing_topics):,} existing entries")
except FileNotFoundError:
    existing_topics = {}
    print("Starting fresh")

# Get all unique author IDs from the extended cohort raw data
combined = pd.read_csv("extended_cohort_raw.csv")
all_authors = combined["author_openalex_id"].unique().tolist()
to_fetch = [aid for aid in all_authors if aid not in existing_topics]
print(f"Total authors: {len(all_authors):,}, need to fetch: {len(to_fetch):,}")

# Run the fetch
if to_fetch:
    all_topics = fetch_all_authors_batched(to_fetch, existing=existing_topics)
else:
    print("Nothing to fetch, all authors already have entries")

Resuming from 972 existing entries
Total authors: 6,722, need to fetch: 5,750
  [    0/5750] saved checkpoint (1,022 total)
  [ 1000/5750] saved checkpoint (2,022 total)
  [ 2000/5750] saved checkpoint (3,022 total)
  [ 3000/5750] saved checkpoint (4,022 total)
  [ 4000/5750] saved checkpoint (5,022 total)
  [ 5000/5750] saved checkpoint (6,022 total)
Done. Total authors with metadata: 6,722


In [12]:
with open("author_topics.json") as f:
    topics = json.load(f)
print(f"Total entries: {len(topics)}")
print(f"Agarwal in file: {'A5089305017' in topics}")

Total entries: 6722
Agarwal in file: True
